In [ ]:
import concurrent.futures
import csv
import os
import sys
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np


plt.rcParams["axes.formatter.useoffset"] = False

root_dir = os.path.abspath("..")
if root_dir not in sys.path:
    sys.path.append(root_dir)

from current_setpoints.models.machines import ieee_machine2_trained_neural_flux
from current_setpoints.models.forward_model import ForwardModel, Fault, count_peaks_at_limit
from current_setpoints.optimization import FeasibilityOptimizer

CURR_MAX, VOLT_MAX = 30.0, 13.0
OMEGA_MAX = 1800.0
N_GRID_POINTS = 6
N_THETA_GRID = 90


POINT_TIMEOUT_SECONDS = 900

PROBE_TIMEOUT_SECONDS = 180

drive = ieee_machine2_trained_neural_flux(curr_max=CURR_MAX, volt_max=VOLT_MAX)
CONST_MECH_SPEED = 30.0 / (np.pi * drive.n_ppairs)
PHASE_LETTERS = ["a", "b", "c", "d", "e"]


FAULT_MODES = {
    "healthy": (),
    "1-fault": (4,),
    "2-adj": (3, 4),
    "2-non-adj": (2, 4),
}
FWD_BY_MODE = {name: ForwardModel(drive, fault=Fault(open_phases), n_theta=700) for name, open_phases in FAULT_MODES.items()}

print(f"const_mech_speed = {CONST_MECH_SPEED:.4f}")

In [ ]:
def segment_label(n_curr, n_volt):
    roman = {0: "", 1: "I", 2: "II"}
    return f"MTPA_{n_curr}" if n_volt == 0 else f"FW_{roman[n_volt]},{n_curr}"


def fine_idx_grid(fwd, theta_grid):
    n_theta = fwd.vec_theta.size - 1
    return np.round(theta_grid / (2 * np.pi) * n_theta).astype(int) % n_theta


def reconstruct_phase_currents(fwd, theta_grid, sched):
    idx_grid = fine_idx_grid(fwd, theta_grid)
    n_surv = fwd.phase_map_at_theta(0).shape[0]
    out = np.full((n_surv, len(idx_grid)), np.nan)
    for k, ti in enumerate(idx_grid):
        if np.all(np.isfinite(sched[k])):
            out[:, k] = fwd.phase_map_at_theta(int(ti)) @ sched[k]
    return out


def reconstruct_phase_voltages(fwd, theta_grid, sched, omega):
    idx_grid = fine_idx_grid(fwd, theta_grid)
    kept = list(fwd.fault._kept)
    out = np.full((len(kept), len(idx_grid)), np.nan)
    for k, ti in enumerate(idx_grid):
        i_n = sched[k]
        if np.all(np.isfinite(i_n)):
            gU, _, bV = fwd.volt_map_at_theta(int(ti), omega, i_n)
            out[:, k] = (gU @ i_n + bV)[kept]
    return out


def alpha_beta_xy(theta_grid, sched):
    c1, s1 = np.cos(theta_grid), np.sin(theta_grid)
    c3, s3 = np.cos(3 * theta_grid), np.sin(3 * theta_grid)
    id1, iq1, id3, iq3 = sched[:, 0], sched[:, 1], sched[:, 2], sched[:, 3]
    i_alpha = c1 * id1 - s1 * iq1
    i_beta = s1 * id1 + c1 * iq1
    i_x = c3 * id3 - s3 * iq3
    i_y = s3 * id3 + c3 * iq3
    return i_alpha, i_beta, i_x, i_y


def save_setpoint_plot(fwd, sol, torque, rpm, omega, save_path, mode):
    """Same 9-subplot figure as pointwise_waveform_diagnostics.ipynb's
    plot_point, but saved to disk instead of shown -- for use in an
    unattended overnight sweep."""
    sched = sol.curr_dq
    theta_grid = sol.diagnostics["theta_grid"]
    theta = np.rad2deg(theta_grid)

    iph = reconstruct_phase_currents(fwd, theta_grid, sched)
    vph = reconstruct_phase_voltages(fwd, theta_grid, sched, omega)
    i_alpha, i_beta, i_x, i_y = alpha_beta_xy(theta_grid, sched)

    n_curr = count_peaks_at_limit(np.concatenate([iph, iph[:, :1]], axis=1), CURR_MAX)
    n_volt = count_peaks_at_limit(np.concatenate([vph, vph[:, :1]], axis=1), VOLT_MAX)
    seg_text = segment_label(n_curr, n_volt)

    kept = list(fwd.fault._kept)
    letters = [PHASE_LETTERS[p] for p in kept]

    norm_pw = np.sum(sched ** 2, axis=1)
    cu_pw = np.nansum(iph ** 2, axis=0)
    torque_pw = np.array([
        fwd.drive.torque(omega, sched[k]) if np.all(np.isfinite(sched[k])) else np.nan
        for k in range(sched.shape[0])
    ])

    have_fft = np.all(np.isfinite(iph))
    if have_fft:
        iph_fft = np.abs(np.fft.rfft(iph, axis=1)) / iph.shape[1]
        harmonics = np.arange(iph_fft.shape[1])

    fig, ax = plt.subplots(3, 3, figsize=(18, 13))
    fig.suptitle(f"{mode}  [{seg_text}]   T*={torque:.2f} Nm,  {rpm:.1f} RPM", fontsize=13)

    dq_lbl = [r"$i_{sd}^1$", r"$i_{sq}^1$", r"$i_{sd}^3$", r"$i_{sq}^3$"]
    dq_col = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
    a = ax[0, 0]
    for j in range(fwd.drive.dim):
        a.plot(theta, sched[:, j], color=dq_col[j], label=dq_lbl[j])
    a.set_title("DQ setpoints")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("DQ current [A]")
    a.legend(fontsize=8)

    a = ax[0, 1]
    for p, lt in enumerate(letters):
        a.plot(theta, iph[p], label=f"$i_{{{lt}}}$")
    a.axhline(CURR_MAX, color="gray", ls=":"); a.axhline(-CURR_MAX, color="gray", ls=":")
    a.set_title("Phase currents")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("phase current [A]")
    a.legend(fontsize=8)

    a = ax[0, 2]
    for p, lt in enumerate(letters):
        a.plot(theta, vph[p], label=f"$v_{{{lt}}}$")
    a.axhline(VOLT_MAX, color="gray", ls=":"); a.axhline(-VOLT_MAX, color="gray", ls=":")
    a.set_title("Phase voltages")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("phase voltage [V]")
    a.legend(fontsize=8)

    a = ax[1, 0]
    a.plot(theta, cu_pw, color="tab:orange", label=f"mean={np.nanmean(cu_pw):.1f}")
    a.set_title(r"Copper-loss density $\sum_k i_k^2(\theta)$")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel(r"$\sum_k i_k^2$ [A$^2$]")
    a.legend(fontsize=8)

    a = ax[1, 1]
    a.plot(theta, norm_pw, color="tab:orange", label=r"$\|i_s(\theta)\|^2$")
    a.set_title(r"DQ norm $\|i_s\|^2$")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel(r"$\|i_s\|^2$ [A$^2$]")
    a.legend(fontsize=8)

    a = ax[1, 2]
    a.plot(i_alpha, i_beta, color="tab:purple")
    a.set_title(r"$\alpha\beta$ current")
    a.set_xlabel(r"$i_\alpha$ [A]"); a.set_ylabel(r"$i_\beta$ [A]")
    a.set_aspect("equal")

    a = ax[2, 0]
    a.plot(i_x, i_y, color="tab:brown")
    a.set_title("xy current")
    a.set_xlabel(r"$i_x$ [A]"); a.set_ylabel(r"$i_y$ [A]")
    a.set_aspect("equal")

    a = ax[2, 1]
    a.plot(theta, torque_pw, color="tab:red", label=f"mean={np.nanmean(torque_pw):.3f}")
    a.axhline(torque, color="gray", ls=":", label=f"T*={torque:.2f}")
    a.set_title("Achieved torque (per-node constraint check)")
    a.set_xlabel(r"$\theta$ [deg]"); a.set_ylabel("torque [Nm]")
    a.legend(fontsize=8)

    a = ax[2, 2]
    if have_fft:
        width = 0.8 / len(letters)
        for p, lt in enumerate(letters):
            a.bar(harmonics + p * width, iph_fft[p], width=width, label=f"$i_{{{lt}}}$")
        a.set_xlim(-0.5, min(15, harmonics[-1]) + 0.5)
        a.set_title("Phase current spectrum (electrical harmonics)")
        a.set_xlabel("harmonic order"); a.set_ylabel("amplitude [A]")
        a.legend(fontsize=8)
    else:
        a.axis("off")
        a.text(0.5, 0.5, "spectrum skipped:\nsome nodes infeasible/NaN",
               ha="center", va="center", transform=a.transAxes)

    plt.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(save_path, dpi=110)
    plt.close(fig)

In [ ]:
def call_with_timeout(timeout, fn, *args, **kwargs):
    """Run fn(*args, **kwargs) with a hard wall-clock cap. If it doesn't
    finish in time, returns (None, "timeout") and moves on -- the
    underlying SLSQP call keeps running in its own thread in the
    background (Windows has no reliable way to kill a specific blocking
    C-extension call from another thread), but the sweep loop itself is
    never blocked past `timeout` seconds by any single setpoint. This is
    the actual fix for "point N took 90+ minutes and never returned" --
    tuning iteration counts alone doesn't provide a hard guarantee, this
    does.
    """
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    future = executor.submit(fn, *args, **kwargs)
    try:
        result = future.result(timeout=timeout)
        executor.shutdown(wait=False)
        return result, None
    except concurrent.futures.TimeoutError:
        executor.shutdown(wait=False)
        return None, "timeout"
    except Exception as exc:
        executor.shutdown(wait=False)
        return None, f"exception: {exc!r}"


def probe_t_max(opt, omega, guess=None):
    """Robust T_max probe: opt.maximize_torque() picks t_max as the min,
    across all angle-nodes, of each node's OWN individually-maximal
    torque -- then asks minimize_current() to hit that value EXACTLY,
    which is precisely the tangent point of that node's own feasible
    boundary (a notoriously hard spot for SLSQP to converge to exactly;
    observed directly: 2/90 nodes failed to converge right at that
    boundary). Back the target off by a small margin, retrying with a
    larger backoff if it still doesn't converge, so the probe lands just
    inside the feasible region instead of exactly on its edge -- this
    doesn't change the physical torque ceiling in any way that matters
    for an 11x11 map (well under 1% at the largest backoff tried).

    Returns (t_max, curr_dq, status) where status is one of:
      "ok"           -- converged, t_max/curr_dq are usable
      "physical"     -- R0's own per-node torque ceiling is <= 0, i.e. the
                        machine genuinely cannot produce positive torque at
                        this speed under this fault
      "no_converge"  -- the ceiling IS positive but no backoff margin
                        converged; the row is NOT known to be infeasible

    Those last two must stay distinct: measured directly that healthy mode
    at 1800 RPM has a POSITIVE ceiling (4.199 Nm) yet fails every backoff,
    so collapsing them would stamp a reachable row "physically infeasible".

    Each backoff attempt is capped at PROBE_TIMEOUT_SECONDS, not the
    larger POINT_TIMEOUT_SECONDS -- this loop already retries up to 6
    margins on its own, so reusing the big per-setpoint timeout here
    would let one probe cost up to 6x that whenever several margins fail.
    """
    maps = opt._precompute_maps(omega)
    r0_grid, ok0 = opt._run_per_angle(omega, maps, None, maximize=True)
    if not np.any(ok0):
        return None, None, "no_converge"
    torques = np.array([opt.fwd.drive.torque(omega, r0_grid[i]) if ok0[i] else np.nan for i in range(opt.n_grid)])
    t_max_raw = float(np.nanmin(torques))

    if t_max_raw <= 0.0:
        return None, None, "physical"

    for margin in (1.0, 0.999, 0.995, 0.99, 0.95, 0.9):
        sol, err = call_with_timeout(PROBE_TIMEOUT_SECONDS, opt.minimize_current, t_max_raw * margin, omega, guess=guess)
        if sol is not None and sol.success:
            return sol.torque, sol.curr_dq, "ok"
    return None, None, "no_converge"


LOG_FIELDS = ["torque", "rpm", "success", "reason", "seconds"]


def append_log_row(log_path, row):
    """Write one row immediately (not buffered until the end) -- a prior
    run was killed mid-sweep by a forced Windows update, losing an
    end-of-run-only log entirely even though the PNGs themselves (written
    incrementally already) survived. Header written once, on first row."""
    is_new = not os.path.exists(log_path)
    with open(log_path, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=LOG_FIELDS)
        if is_new:
            writer.writeheader()
        writer.writerow(row)


def run_sweep(ftol, mode, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    log_path = os.path.join(out_dir, "sweep_log.csv")
    solver_opts = {"disp": False, "ftol": ftol, "maxiter": 300}
    fwd = FWD_BY_MODE[mode]
    opt = FeasibilityOptimizer(fwd, solver_opts, n_grid=N_THETA_GRID)

    omega_probe = 0.0
    torq_max_global, _, status_global = probe_t_max(opt, omega_probe)
    if torq_max_global is None:
        print(f"[{mode}] WARNING: global T_max probe at omega=0 returned '{status_global}' -- aborting this sweep.")
        return []

    vec_torq = np.linspace(0.0, torq_max_global, N_GRID_POINTS)
    vec_omega = np.linspace(0.0, OMEGA_MAX / CONST_MECH_SPEED, N_GRID_POINTS)

    log_rows = []
    prev_max_curr = None
    n_skipped = 0

    for idx_omega, omega in enumerate(vec_omega):
        omega = float(omega)
        rpm = omega * CONST_MECH_SPEED

        torq_max_local, curr_at_max, status = probe_t_max(opt, omega, guess=prev_max_curr)

        if status == "physical":
            print(f"[{mode}] speed {idx_omega + 1}/{len(vec_omega)}: {rpm:.1f} RPM  "
                  f"-- torque ceiling is non-positive, skipping row (physically infeasible)")
            for torque in vec_torq:
                row = {"torque": float(torque), "rpm": rpm, "success": False,
                       "reason": "physically infeasible: R0 torque ceiling <= 0", "seconds": 0.0}
                log_rows.append(row)
                append_log_row(log_path, row)
            continue

        if torq_max_local is None:
            print(f"[{mode}] speed {idx_omega + 1}/{len(vec_omega)}: {rpm:.1f} RPM  "
                  f"-- T_max probe did not converge, attempting row with global T_max~{torq_max_global:.2f} Nm")
            torq_max_local = torq_max_global
        else:
            prev_max_curr = curr_at_max
            print(f"[{mode}] speed {idx_omega + 1}/{len(vec_omega)}: {rpm:.1f} RPM  (T_max~{torq_max_local:.2f} Nm)")

        prev_curr = None
        for idx_torq, torque in enumerate(vec_torq):
            torque = float(torque)
            fname = f"T{torque:06.2f}_RPM{rpm:07.1f}.png".replace(" ", "0")
            fpath = os.path.join(out_dir, fname)

            if os.path.exists(fpath):
                row = {"torque": torque, "rpm": rpm, "success": True, "reason": "already done (resumed run)", "seconds": 0.0}
                log_rows.append(row)
                append_log_row(log_path, row)
                n_skipped += 1
                print(f"    T={torque:.2f} Nm  already done, skipping")
                continue

            if torque > min(torq_max_local, torq_max_global):
                row = {"torque": torque, "rpm": rpm, "success": False, "reason": "above T_max, skipped", "seconds": 0.0}
                log_rows.append(row)
                append_log_row(log_path, row)
                continue

            t0 = time.perf_counter()
            sol, err = call_with_timeout(POINT_TIMEOUT_SECONDS, opt.minimize_current, torque, omega, guess=prev_curr)
            dt = time.perf_counter() - t0

            if sol is None:
                row = {"torque": torque, "rpm": rpm, "success": False, "reason": err, "seconds": dt}
                log_rows.append(row)
                append_log_row(log_path, row)
                print(f"    T={torque:.2f} Nm  {err}  ({dt:.1f}s)")
                continue

            if sol.success:
                prev_curr = sol.curr_dq
                try:
                    save_setpoint_plot(fwd, sol, torque, rpm, omega, fpath, mode)
                    row = {"torque": torque, "rpm": rpm, "success": True, "reason": "", "seconds": dt}
                    print(f"    T={torque:.2f} Nm  OK  ({dt:.1f}s)")
                except Exception as exc:
                    row = {"torque": torque, "rpm": rpm, "success": False, "reason": f"plot exception: {exc!r}", "seconds": dt}
                    print(f"    T={torque:.2f} Nm  CONVERGED BUT PLOT FAILED: {exc!r}  ({dt:.1f}s)")
            else:
                reason = sol.diagnostics.get("fail_reason", "active-set voltage feasibility did not converge")
                row = {"torque": torque, "rpm": rpm, "success": False, "reason": reason, "seconds": dt}
                print(f"    T={torque:.2f} Nm  FAILED: {reason}  ({dt:.1f}s)")

            log_rows.append(row)
            append_log_row(log_path, row)

    n_ok = sum(1 for r in log_rows if r["success"])
    print(f"\n[{mode}] done: {n_ok}/{len(log_rows)} points saved ({n_skipped} already done from a prior run), log at {log_path}")
    return log_rows

In [ ]:
FTOL = 1e-8
results_by_mode = {}
for mode in FAULT_MODES:
    results_by_mode[mode] = run_sweep(ftol=FTOL, mode=mode, out_dir=f"../figs/feasibility_sweep_{mode}")

In [ ]:
for mode, results in results_by_mode.items():
    n_ok = sum(1 for r in results if r["success"])
    print(f"{mode}: {n_ok}/{len(results)} setpoints converged and were plotted")

In [ ]:
def read_sweep_log(mode, out_dir=None):
    """Read one mode's sweep log back off disk. Reads the CSV rather than the
    in-memory results so the map can be drawn WHILE a sweep is still running
    (rows are flushed one at a time), and after an interrupted or resumed
    run. Later rows win on duplicate (rpm, torque) keys, which is what a
    resumed run produces."""
    out_dir = out_dir if out_dir is not None else f"../figs/feasibility_sweep_{mode}"
    log_path = os.path.join(out_dir, "sweep_log.csv")
    if not os.path.exists(log_path):
        return {}
    points = {}
    with open(log_path, newline="") as f:
        for row in csv.DictReader(f):
            key = (round(float(row["rpm"]), 3), round(float(row["torque"]), 6))
            points[key] = {
                "rpm": float(row["rpm"]),
                "torque": float(row["torque"]),
                "success": row["success"] == "True",
                "reason": row["reason"],
                "seconds": float(row["seconds"]) if row["seconds"] else 0.0,
            }
    return points


def plot_feasibility_maps(modes=None, save_path="../figs/feasibility_map.png"):
    """Torque/speed map per mode: green = a feasible trajectory was found and
    verified, red = not. 'Not' bundles genuinely infeasible points together
    with ones that were never attempted (above the T_max ceiling, or a speed
    row skipped as physically infeasible) -- those carry a hollow marker so a
    real solver failure is never mistaken for a point nobody tried."""
    modes = list(FAULT_MODES) if modes is None else list(modes)
    n = len(modes)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7.0 * ncols, 5.0 * nrows), squeeze=False)

    not_attempted = ("above T_max, skipped", "physically infeasible: R0 torque ceiling <= 0")

    for idx, mode in enumerate(modes):
        ax = axes[idx // ncols][idx % ncols]
        pts = read_sweep_log(mode)
        if not pts:
            ax.text(0.5, 0.5, f"{mode}\n(no log yet)", ha="center", va="center", transform=ax.transAxes)
            ax.set_xticks([]); ax.set_yticks([])
            continue

        ok = [(p["rpm"], p["torque"]) for p in pts.values() if p["success"]]
        tried = [(p["rpm"], p["torque"]) for p in pts.values() if not p["success"] and p["reason"] not in not_attempted]
        skipped = [(p["rpm"], p["torque"]) for p in pts.values() if not p["success"] and p["reason"] in not_attempted]

        if ok:
            ax.scatter(*zip(*ok), c="tab:green", s=110, marker="o",
                       edgecolors="black", linewidths=0.5, label=f"feasible ({len(ok)})", zorder=3)
        if tried:
            ax.scatter(*zip(*tried), c="tab:red", s=110, marker="X",
                       edgecolors="black", linewidths=0.5, label=f"infeasible ({len(tried)})", zorder=3)
        if skipped:
            ax.scatter(*zip(*skipped), facecolors="none", edgecolors="tab:red", s=110, marker="o",
                       linewidths=1.4, label=f"not attempted ({len(skipped)})", zorder=2)

        ax.set_title(f"{mode}   {len(ok)}/{len(pts)} feasible")
        ax.set_xlabel("speed [RPM]")
        ax.set_ylabel("torque [Nm]")
        ax.grid(alpha=0.3, zorder=0)
        ax.legend(fontsize=8, loc="upper right")

    for idx in range(n, nrows * ncols):
        axes[idx // ncols][idx % ncols].axis("off")

    fig.suptitle("Feasible setpoints (dynamic mode, exact voltage residual verified)", fontsize=13)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(save_path, dpi=110)
    plt.close(fig)
    print(f"saved {save_path}")
    for mode in modes:
        pts = read_sweep_log(mode)
        n_ok = sum(1 for p in pts.values() if p["success"])
        print(f"  {mode:11s} {n_ok:3d}/{len(pts):3d} feasible")


plot_feasibility_maps()